# Projeto 8 — Classificação de Ataques de Rede (NSL-KDD) com Random Forest e SVM

Notebook principal para comparação didática entre **Random Forest** e **SVM** com pipeline de pré-processamento, seleção de características, balanceamento e métricas completas.

## 1) Dataset e organização de pastas

- Dataset oficial (NSL-KDD): https://www.unb.ca/cic/datasets/nsl.html
- Coloque os arquivos abaixo em `data/raw/`:
  - `KDDTrain+.txt`
  - `KDDTest+.txt`

Este notebook lê apenas a partir de `data/raw/`, conforme requisito do projeto.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data import load_nsl_kdd_dataset, split_features_target, infer_feature_types
from src.models import build_rf_pipeline, build_svm_pipeline, rf_param_grid, svm_param_grid, build_search, optional_smote
from src.evaluation import time_fit_and_predict, classification_metrics, plot_confusion, summarize_result

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

In [ ]:
raw_dir = ROOT / 'data' / 'raw'
train_df, test_df = load_nsl_kdd_dataset(raw_dir=raw_dir)

print('Treino:', train_df.shape)
print('Teste :', test_df.shape)
display(train_df.head())

In [ ]:
X_train, y_train = split_features_target(train_df, target_mode='category')
X_test, y_test = split_features_target(test_df, target_mode='category')

cat_cols, num_cols = infer_feature_types(X_train)
print('Colunas categóricas:', cat_cols)
print('Qtd. colunas numéricas:', len(num_cols))

print('Distribuição de classes (treino):')
display(y_train.value_counts())

## 2) Modelos e estratégia experimental

- **Random Forest** e **SVM** com `Pipeline`
- Pré-processamento via `ColumnTransformer`
- Seleção de características: `SelectKBest(mutual_info_classif)` comparada com `passthrough`
- Balanceamento: `class_weight='balanced'`
- Busca de hiperparâmetros: `GridSearchCV`

In [ ]:
rf_pipe = build_rf_pipeline(cat_cols, num_cols, random_state=RANDOM_STATE)
svm_pipe = build_svm_pipeline(cat_cols, num_cols, random_state=RANDOM_STATE)

rf_search = build_search(rf_pipe, rf_param_grid(RANDOM_STATE), cv=3, scoring='f1_weighted', n_jobs=-1)
svm_search = build_search(svm_pipe, svm_param_grid(), cv=3, scoring='f1_weighted', n_jobs=-1)

print('RF e SVM prontos para busca de hiperparâmetros.')

In [ ]:
labels = sorted(pd.unique(pd.concat([y_train, y_test], axis=0)))

rf_pred, rf_train_t, rf_inf_t = time_fit_and_predict(rf_search, X_train, y_train, X_test)
rf_metrics = classification_metrics(y_test, rf_pred, labels=labels)

print('Melhores hiperparâmetros RF:')
print(rf_search.best_params_)
print('\nClassification report RF:')
print(rf_metrics['classification_report_text'])

In [ ]:
plot_confusion(rf_metrics['confusion_matrix'], labels, title='Random Forest - Matriz de Confusão')
plt.show()

In [ ]:
svm_pred, svm_train_t, svm_inf_t = time_fit_and_predict(svm_search, X_train, y_train, X_test)
svm_metrics = classification_metrics(y_test, svm_pred, labels=labels)

print('Melhores hiperparâmetros SVM:')
print(svm_search.best_params_)
print('\nClassification report SVM:')
print(svm_metrics['classification_report_text'])

In [ ]:
plot_confusion(svm_metrics['confusion_matrix'], labels, title='SVM - Matriz de Confusão')
plt.show()

In [ ]:
summary = pd.DataFrame([
    summarize_result('Random Forest', rf_metrics, rf_train_t, rf_inf_t),
    summarize_result('SVM', svm_metrics, svm_train_t, svm_inf_t),
]).sort_values(by='f1_weighted', ascending=False)

display(summary)

## 3) Experimento opcional com SMOTE (fallback automático)

Este passo é opcional. Caso `imbalanced-learn` não esteja instalado, a célula informa e segue com os resultados base (`class_weight`).

In [ ]:
smote = optional_smote(random_state=RANDOM_STATE)
if smote is None:
    print('imbalanced-learn não instalado. Experimento SMOTE ignorado (fallback OK).')
else:
    try:
        from imblearn.pipeline import Pipeline as ImbPipeline
        from sklearn.ensemble import RandomForestClassifier
        from src.models import build_preprocessor

        pre = build_preprocessor(cat_cols, num_cols, scale_numeric=False)
        rf_smote = ImbPipeline(steps=[
            ('preprocessor', pre),
            ('smote', smote),
            ('classifier', RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ])

        y_pred_smote, tr_sm, inf_sm = time_fit_and_predict(rf_smote, X_train, y_train, X_test)
        m_sm = classification_metrics(y_test, y_pred_smote, labels=labels)
        print(m_sm['classification_report_text'])
        print(f'Tempo treino: {tr_sm:.3f}s | inferência: {inf_sm:.3f}s')
    except Exception as exc:
        print(f'Não foi possível executar SMOTE neste ambiente: {exc}')

## 4) Conclusões para apresentação

- Comparar a tabela de métricas (acurácia e F1 ponderado).
- Comparar matriz de confusão por classe (normal, dos, probe, r2l, u2r).
- Discutir trade-off de tempo: RF normalmente treina mais rápido; SVM pode variar conforme kernel e C.
- Destacar que a seleção de características (`SelectKBest`) foi avaliada dentro da busca de hiperparâmetros.